In [ ]:
import cv2
import face_recognition
import pickle
import pandas as pd
from datetime import datetime
import os
import numpy as np
import time

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
# Using 'r' before the string to handle Windows backslashes correctly
ENCODINGS_PATH = r"D:\SKILLPARK\SMART_CLASSROOOM\encodings.pkl"
SAVE_PATH = r"D:\SKILLPARK\SMART_CLASSROOOM"

# THRESHOLD: Lower is stricter. 
# 0.6 is the library default. 0.5 is better for avoiding false positives.
THRESHOLD = 0.5  

# Ensure the directory for saving Excel files exists
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# ==========================================
# 2. LOAD DATABASE
# ==========================================
print("📂 Loading facial database...")
try:
    with open(ENCODINGS_PATH, "rb") as f:
        data = pickle.load(f)
    known_encodings = data["encodings"]
    known_names = data["names"]
    print(f"✅ Loaded {len(known_names)} facial signatures.")
except FileNotFoundError:
    print(f"❌ Error: {ENCODINGS_PATH} not found.")
    print("Please ensure your encoding script has run and saved the .pkl file.")
    exit()

# Tracking variables for the current session
present_students = set()
attendance_log = []

# ==========================================
# 3. LIVE CAMERA LOOP
# ==========================================
print("📷 Initializing camera...")

# cv2.CAP_DSHOW is a Windows-specific fix to open the camera faster and more reliably
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW) 

# Give the camera hardware a 2-second "warm-up" to adjust light and focus
time.sleep(2)

if not cap.isOpened():
    print("❌ Error: Could not open webcam. Ensure no other app is using it.")
    exit()

print("🚀 System Active! Press 'q' to SAVE and EXIT.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("❌ Failed to grab frame. Retrying...")
        continue

    # Step A: Convert BGR (OpenCV default) to RGB (face_recognition requirement)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Step B: Find all face locations and encodings in the current frame
    face_locations = face_recognition.face_locations(rgb_frame)
    face_encodings = face_recognition.face_encodings(rgb_frame, face_locations)

    for face_encoding, face_loc in zip(face_encodings, face_locations):
        # Step C: Compare the live face to our loaded database
        # This returns an array of distances (how different they are)
        face_distances = face_recognition.face_distance(known_encodings, face_encoding)
        best_match_index = np.argmin(face_distances)
        
        # Lower distance means a better match (0.0 is identical)
        score = round(face_distances[best_match_index], 2)
        
        name = "Unknown"
        color = (0, 0, 255) # Red for unknown

        # Step D: Match Logic based on Threshold
        if score < THRESHOLD:
            name = known_names[best_match_index]
            color = (0, 255, 0) # Green for recognized

            # --- PREVENT DUPLICATE LOGS ---
            if name not in present_students:
                now = datetime.now()
                present_students.add(name)
                attendance_log.append({
                    "Student Name": name,
                    "Date": now.strftime("%Y-%m-%d"),
                    "Time": now.strftime("%H:%M:%S"),
                    "Distance Score": score
                })
                print(f"✔️ MARKED PRESENT: {name} (Score: {score})")

        # Step E: VISUAL HUD (Draw bounding box and label)
        top, right, bottom, left = face_loc
        
        # Main bounding box
        cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
        
        # Label background
        cv2.rectangle(frame, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
        
        # Text for Name and Distance Score
        display_text = f"{name} ({score})"
        cv2.putText(frame, display_text, (left + 6, bottom - 6), 
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)

    # Step F: On-screen session summary
    cv2.putText(frame, f"Logged Today: {len(present_students)}", (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    # Display the resulting image
    cv2.imshow("Smart Classroom Attendance System", frame)

    # Break loop on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ==========================================
# 4. EXPORT TO EXCEL
# ==========================================
# Release camera and close windows before processing the file
cap.release()
cv2.destroyAllWindows()

if attendance_log:
    df = pd.DataFrame(attendance_log)
    
    # Create a unique filename based on the current date and time
    file_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    excel_file = f"Attendance_Log_{file_timestamp}.xlsx"
    full_save_path = os.path.join(SAVE_PATH, excel_file)
    
    # Save the data
    df.to_excel(full_save_path, index=False)
    
    print("\n" + "="*40)
    print(f"🏁 SESSION COMPLETE")
    print(f"📁 File Saved: {full_save_path}")
    print(f"👥 Total Students Marked: {len(present_students)}")
    print("="*40)
else:
    print("\n⚠️ Session closed. No attendance was recorded.")

📂 Loading facial database...
✅ Loaded 1081 facial signatures.
❌ Failed to grab frame.

⚠️ Session closed. No attendance was recorded.
